In [1]:
import numpy  as np
import pandas as pd

pd.options.mode.chained_assignment = None  # default='warn'

# Tester la qualité des prédiction

+ Divisé le dataframe :
    * 'train_df' avec 6000 lignes dr 'listings' ;
    * 'test_df' avec le reste.
* Modifier la fonction, changer la datagrame 'temp_df'. Idem avec le dataframe 'listings' à 'train_df' pour que seul le dataset de training soit utilisé ;
* Utiliser la méthode '`Series apply`' pour appliquer la fonction 'predict_price' sur les valeurs de la colonne 'accomodates' du dataframe de test ;
* Assigner l'objet '`Series`' résultant à la colonnes 'predicted_price' de 'test_df'.

In [2]:
listings          = pd.read_csv('paris_airbnb.csv')
stripped_commas   = listings['price'].str.replace(',', '')
stripped_dollars  = stripped_commas.str.replace('$', '')
listings['price'] = stripped_dollars.astype('float')

In [3]:
train_df = listings.iloc[:6000]
test_df  = listings.iloc[6000:]

In [4]:
# 80% train / 20% test
train_df = listings.sample(frac=0.75, random_state=42)
test_df  = listings.drop(train_df.index)

print(f"Il y a {len(train_df)} logements dans le jeu d'entraînement \net {len(test_df)} logements dans le jeu de test.")

Il y a 6000 logements dans le jeu d'entraînement 
et 2000 logements dans le jeu de test.


In [5]:
def predict_price(new_listing):
    temp_df             = train_df.copy()
    temp_df['distance'] = temp_df['accommodates'].apply(lambda x: np.abs(x - new_listing))
    temp_df             = temp_df.sort_values('distance')
    nearest_neighbors   = temp_df.iloc[0:5]['price']
    predicted_price     = nearest_neighbors.mean()
    return(predicted_price)

In [6]:
test_df['predicted_price'] = test_df['accommodates'].apply(lambda x: predict_price(x))

### Les métriques d'erreur

* Utiliser la méthode '`numpy.absolute()`' pour calculer l'erreur absolue moyenne MAE entre 'predicted_price' et 'price' ;
* Assigner le résultat à la variable MAE.

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \left| y_i - \hat{y}_i \right|
= \frac{|y_1 - \hat{y}_1| + |y_2 - \hat{y}_2| + \cdots + |y_n - \hat{y}_n|}{n}
$$

In [7]:
test_df.head()

,host_response_rate,host_acceptance_rate,host_listings_count,latitude,longitude,city,zipcode,state,accommodates,room_type,bedrooms,bathrooms,beds,price,cleaning_fee,security_deposit,minimum_nights,maximum_nights,number_of_reviews,predicted_price
2,100%,NaN,2.0,48.85758,2.35275,Paris,75004,Île-de-France,4,Entire home/apt,2.0,1.0,2.0,115.0,$50.00,$200.00,10,23,243,77.0
3,100%,NaN,1.0,48.86528,2.39326,Paris,75020,Ile-de-France,3,Entire home/apt,1.0,1.0,1.0,90.0,NaN,NaN,3,365,1,93.8
4,67%,NaN,3.0,48.85899,2.34735,Paris,75001,Île-de-France,2,Entire home/apt,1.0,1.0,1.0,75.0,$200.00,"$1,500.00",180,365,0,80.8
5,100%,NaN,1.0,48.86227,2.37134,Paris,75011,Ile-de-France,2,Entire home/apt,1.0,1.0,1.0,75.0,$20.00,$250.00,5,120,17,80.8
9,100%,NaN,2.0,48.84670,2.35095,Paris,75005,Île-de-France,2,Entire home/apt,0.0,1.0,1.0,75.0,NaN,NaN,2,1125,336,80.8


In [8]:
test_df['error'] = np.abs(test_df['predicted_price'] - test_df['price'])
mae = test_df['error'].mean()
print(f"Le MAE du modèle est de {mae:.2f} $.")

Le MAE du modèle est de 46.48 $.


### Erreur quadratique moyenne (Mean Squared Error)

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2
= \frac{(y_1 - \hat{y}_1)^2 + (y_2 - \hat{y}_2)^2 + \cdots + (y_n - \hat{y}_n)^2}{n}
$$

* Calculer la valeur MSE entre les colonnes 'predicted_price' et 'price' ;
* L'assigner à la MSE ;
* L'afficher

Le point crucial à retenir est que la MSE amplifie les erreurs importantes quadratiquement, ce qui la rend plus sévère que la MAE envers les valeurs aberrantes. C'est pourquoi la RMSE (racine carrée de la MSE) est souvent préférée pour l'interprétabilité, car elle ramène le résultat dans l'unité d'origine de \(y\)

In [9]:
test_df['squared_error'] = (test_df['predicted_price'] - test_df['price']) ** 2
mse = test_df['squared_error'].mean()
print(f"Le MSE du modèle est de {mse:.2f} $.")

Le MSE du modèle est de 8660.06 $.
